# UNSW-NB15: network-flow extraction with NFStream

This notebook extracts bidirectional network flows from the original UNSW-NB15 PCAP files using NFStream. The PCAP captures are organized into two groups corresponding to 22 January 2015 and 17 February 2015. Each group is consolidated into one CSV file for subsequent labeling and customization in the GenIDS-NB15 preparation notebook.

Only the paths and, if necessary, the PCAP filename lists in the configuration cell should be changed. The original PCAP files are not distributed with this repository.

## 1. Configuration

In [ ]:
from pathlib import Path

PCAP_JANUARY_DIR = Path("/path/to/unsw-nb15/pcap_22_01_15")
PCAP_FEBRUARY_DIR = Path("/path/to/unsw-nb15/pcap_17_02_15")
OUTPUT_DIR = Path("/path/to/output/nfstream_flows")

JANUARY_OUTPUT_FILE = OUTPUT_DIR / "pcap1.csv"
FEBRUARY_OUTPUT_FILE = OUTPUT_DIR / "pcap2.csv"

JANUARY_PCAP_FILES = [f"{index:02}.pcap" for index in range(1, 54)]
FEBRUARY_PCAP_FILES = [f"{index:02}.pcap" for index in range(1, 27)]

IDLE_TIMEOUT = 300
ACTIVE_TIMEOUT = 20
STATISTICAL_ANALYSIS = True
DECODE_TUNNELS = True
BPF_FILTER = 'ip'

The January group contains the files `01.pcap` through `53.pcap`. The February group uses 26 sequentially named files. In the original processing, a problematic capture was discarded and the subsequent February files were renamed before extraction; therefore, the configuration reflects the final sequence `01.pcap` through `26.pcap`.

## 2. Imports and helper functions

In [ ]:
import pandas as pd
import nfstream
from nfstream import NFStreamer


def validate_pcap_files(directory, filenames, group_name):
    if not directory.is_dir():
        raise NotADirectoryError(f"{group_name} PCAP directory not found: {directory}")
    missing = [directory / filename for filename in filenames if not (directory / filename).is_file()]
    if missing:
        formatted = "\n".join(f"- {path}" for path in missing)
        raise FileNotFoundError(f"Missing {group_name} PCAP files:\n{formatted}")


def extract_pcap_file(path):
    return NFStreamer(
        source=str(path),
        idle_timeout=IDLE_TIMEOUT,
        active_timeout=ACTIVE_TIMEOUT,
        statistical_analysis=STATISTICAL_ANALYSIS,
        decode_tunnels=DECODE_TUNNELS,
        bpf_filter=BPF_FILTER,
    ).to_pandas()


def extract_pcap_group(directory, filenames, group_name):
    frames = []
    records = []
    for position, filename in enumerate(filenames, start=1):
        path = directory / filename
        print(f"[{position}/{len(filenames)}] Extracting {group_name}: {filename}")
        flows = extract_pcap_file(path)
        frames.append(flows)
        records.append({
            "group": group_name,
            "pcap_file": filename,
            "extracted_flows": len(flows),
        })
        print(f"    Extracted flows: {len(flows):,}")
    if not frames:
        raise ValueError(f"No PCAP files were configured for {group_name}.")
    return pd.concat(frames, ignore_index=True), pd.DataFrame(records)


def save_flows(frame, output_file):
    output_file.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(output_file, index=False)
    if not output_file.is_file():
        raise OSError(f"The output file was not created: {output_file}")
    print(f"Saved {len(frame):,} flows to {output_file.resolve()}")

## 3. Validate input files

In [ ]:
print(f"NFStream version: {nfstream.__version__}")

validate_pcap_files(PCAP_JANUARY_DIR, JANUARY_PCAP_FILES, "January 2015")
validate_pcap_files(PCAP_FEBRUARY_DIR, FEBRUARY_PCAP_FILES, "February 2015")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input validation completed successfully.")

## 4. Extract flows from the January PCAP files

In [ ]:
january_flows, january_summary = extract_pcap_group(
    PCAP_JANUARY_DIR,
    JANUARY_PCAP_FILES,
    "22 January 2015",
)

save_flows(january_flows, JANUARY_OUTPUT_FILE)
display(january_summary)
print(f"January output shape: {january_flows.shape}")

## 5. Extract flows from the February PCAP files

In [ ]:
february_flows, february_summary = extract_pcap_group(
    PCAP_FEBRUARY_DIR,
    FEBRUARY_PCAP_FILES,
    "17 February 2015",
)

save_flows(february_flows, FEBRUARY_OUTPUT_FILE)
display(february_summary)
print(f"February output shape: {february_flows.shape}")

## 6. Extraction summary

In [ ]:
extraction_summary = pd.DataFrame([
    {
        "group": "22 January 2015",
        "pcap_files": len(JANUARY_PCAP_FILES),
        "extracted_flows": len(january_flows),
        "output_file": str(JANUARY_OUTPUT_FILE.resolve()),
    },
    {
        "group": "17 February 2015",
        "pcap_files": len(FEBRUARY_PCAP_FILES),
        "extracted_flows": len(february_flows),
        "output_file": str(FEBRUARY_OUTPUT_FILE.resolve()),
    },
])

display(extraction_summary)
print(f"Total extracted flows: {len(january_flows) + len(february_flows):,}")